<br>
<br>

# `#01: How to break down a pretrained model`

<br>
<br>

In [13]:

import torch 
import warnings

# torchinfo is model then torchsummary: use torchinfo:
from torchinfo import summary

from torchvision.models import resnet50, ResNet50_Weights


warnings.filterwarnings('ignore')

model = resnet50(ResNet50_Weights.DEFAULT)

# see: here the depth: 
summary(model,input_size=(1,3,224,224),depth=1)



Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 256, 56, 56]          215,808
├─Sequential: 1-6                        [1, 512, 28, 28]          1,219,584
├─Sequential: 1-7                        [1, 1024, 14, 14]         7,098,368
├─Sequential: 1-8                        [1, 2048, 7, 7]           14,964,736
├─AdaptiveAvgPool2d: 1-9                 [1, 2048, 1, 1]           --
├─Linear: 1-10                           [1, 1000]                 2,049,000
Total params: 25,557,032
Trainable params: 25,557,032
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 4.09

In [ ]:

# see: here the depth: (we can break the model info more details)
summary(model,input_size=(1,3,224,224),depth=2)


Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 256, 56, 56]          --
│    └─Bottleneck: 2-1                   [1, 256, 56, 56]          75,008
│    └─Bottleneck: 2-2                   [1, 256, 56, 56]          70,400
│    └─Bottleneck: 2-3                   [1, 256, 56, 56]          70,400
├─Sequential: 1-6                        [1, 512, 28, 28]          --
│    └─Bottleneck: 2-4                   [1, 512, 28, 28]          379,392
│    └─Bottleneck: 2-5                   [1, 512, 28, 28]          280,064
│    └─Bottleneck: 2-6                   [1, 512, 28, 28]  

In [9]:


# 1. We do code like: 
""" 
self.layer1 = Sequential()
self.layer2 = Sequential()
self.layer3 = Sequential()
self.layer4 = Sequential()
"""

for name,module in model.named_children():
    print(name)

# uses of this: 
# see, here we have-> 0,1,2,3,4
print("============================Access a layer=========================")
print(model.layer1)
print()
print(model.layer1[0])
print(model.layer1[0].conv1.weight.shape)



conv1
bn1
relu
maxpool
layer1
layer2
layer3
layer4
avgpool
fc
============================Access a layer=========================
Sequential(
  (0): Bottleneck(
    (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (downsample): Sequential(
      (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
  )
  (1): Bottleneck(
    (conv1): Conv2d(256


<br>
<br>


# `#03: Buffers in PyTorch`

<br>
<br>


## 1. What is a Buffer?

A **buffer** is a tensor that lives inside a model but is **never learned**.

A buffer:
- ❌ Is **not** updated by `loss.backward()` or the optimizer (no gradient)
- ✅ **Is saved** in `state_dict()`, so it is stored when you save the model
- ✅ **Moves with the model** when you call `model.to('cuda')` or `model.to('cpu')`

**Simple idea:** A buffer is the model's memory, not its weights.

🔑 **Golden rule:**
- **Needs to be learned (needs gradient)** → **Parameter**
- **Only needs to be remembered or calculated** → **Buffer**

A buffer **never holds anything learnable**. Some buffers do change in the forward pass, but that change is a simple calculation, not learning.

---

## 2. Parameter vs Buffer vs Normal Tensor

| | Parameter | Buffer | Normal tensor (`self.x = torch.tensor(...)`) |
|---|---|---|---|
| Learns with gradient? | ✅ Yes | ❌ No | ❌ No |
| Seen by the optimizer? | ✅ Yes | ❌ No | ❌ No |
| Saved in `state_dict`? | ✅ Yes | ✅ Yes | ❌ No |
| Moves with `.to(device)`? | ✅ Yes | ✅ Yes | ❌ No |

⚠️ **Common bug:** If you store a fixed tensor as a normal attribute, it stays on the CPU after `model.to('cuda')`, and you get a **device mismatch error**. Use a buffer instead.

---

## 3. Best Example: BatchNorm Has Both

One BatchNorm layer contains **both** parameters and buffers:

| Item | Type | How it changes |
|---|---|---|
| `weight` (γ, gamma) | **Parameter** | **Learns** through `loss.backward()` + optimizer |
| `bias` (β, beta) | **Parameter** | **Learns** through `loss.backward()` + optimizer |
| `running_mean` | **Buffer** | Calculated in the forward pass (not learned) |
| `running_var` | **Buffer** | Calculated in the forward pass (not learned) |
| `num_batches_tracked` | **Buffer** | Just counts +1 for each batch |

**Why is this split?**
- **Mean and variance** are **statistics measured from the data**. Nothing is learned, so they go in **buffers**.
- **Gamma and beta** are **learned values** that scale and shift the output, so they are **parameters**.

Check it yourself:

```python
bn = nn.BatchNorm2d(3)

print([n for n, _ in bn.named_parameters()])
# ['weight', 'bias']                                       ← learnable

print([n for n, _ in bn.named_buffers()])
# ['running_mean', 'running_var', 'num_batches_tracked']   ← not learnable
```

---

## 4. Two Types of Buffers

### (A) Dynamic Buffers: change in the forward pass (but still NOT learned)

- They do **not** learn from the loss. No gradient, no optimizer.
- In **training mode** (`model.train()`), their values are **recalculated automatically** every time a batch passes forward.
- No `loss.backward()` is needed for this update.

**Example:** BatchNorm's `running_mean`, `running_var`, `num_batches_tracked`

**How the update works** (default `momentum = 0.1`):

```
running_mean = (1 - 0.1) × running_mean + 0.1 × batch_mean
running_var  = (1 - 0.1) × running_var  + 0.1 × batch_var
```

This is just a **moving average formula**, not learning. The new batch has a small effect, and the old history keeps most of the weight.

**Why we need them:**
- During **testing** (`model.eval()`), you may give only **one image**.
- One image cannot give a good mean or variance by itself.
- So BatchNorm uses the saved `running_mean` and `running_var` instead.

🔑 In `model.eval()` mode, these buffers **stop updating**. They are only read.

---

### (B) Static Buffers: 100% fixed

- They are created **once** in `__init__`.
- They **never change**, not in the forward pass and not in backpropagation.

**Examples:**

| Buffer | Used in | Purpose |
|---|---|---|
| **Positional Encoding** (fixed sin/cos values) | Transformer, ViT | Tells the model the position of each word or patch |
| **Causal / Attention Mask** (lower triangular matrix) | GPT, LLMs | Hides future tokens so the model can't "cheat" |
| **Anchor Boxes** | YOLO, Faster R-CNN, RetinaNet | Fixed pre-defined boxes for finding objects |
| **Fixed Filters** (Sobel, Gaussian blur) | Custom CNNs | Apply a fixed image filter that never trains |
| **Normalization Mean / Std** (e.g. ImageNet values) | Image models | Normalize input pixels inside the model |

---

## 5. How to Create a Buffer: `register_buffer()`

```python
import torch
import torch.nn as nn

class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Learnable weights (Parameters)
        self.conv = nn.Conv2d(3, 64, kernel_size=3)

        # Static buffer: fixed ImageNet mean
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        self.register_buffer('image_mean', mean)

        # Static buffer that is NOT saved in state_dict
        mask = torch.tril(torch.ones(8, 8))
        self.register_buffer('causal_mask', mask, persistent=False)

    def forward(self, x):
        x = x - self.image_mean   # buffer is already on the same device as x
        return self.conv(x)
```

**`persistent=False`** means:
- The buffer still moves with `.to(device)`.
- But it is **not saved** in `state_dict`.
- Useful for things that are easy to rebuild, like a causal mask.

### How to list buffers

```python
for name, buf in model.named_buffers():
    print(name, buf.shape)
```

---

## 6. Quick Revision Summary ⚡

- **Buffer** = a tensor that is saved and moves with the model, but has **no gradient** and is **never learned**.
- **Learned → Parameter. Measured or fixed → Buffer.**
- **BatchNorm:** gamma and beta = **parameters**; running mean and variance = **buffers**.
- **Dynamic buffer** = recalculated in the forward pass during training (by a formula, not by learning).
- **Static buffer** = never changes (positional encoding, masks, anchors, fixed filters).
- Create one with `self.register_buffer('name', tensor)`.
- Use `persistent=False` if you don't want it saved.
- Never store fixed tensors as plain attributes, or you'll get a device mismatch error.

<br>
<br>



In [10]:

for name, buf in model.named_buffers():
    print(name)
    

bn1.running_mean
bn1.running_var
bn1.num_batches_tracked
layer1.0.bn1.running_mean
layer1.0.bn1.running_var
layer1.0.bn1.num_batches_tracked
layer1.0.bn2.running_mean
layer1.0.bn2.running_var
layer1.0.bn2.num_batches_tracked
layer1.0.bn3.running_mean
layer1.0.bn3.running_var
layer1.0.bn3.num_batches_tracked
layer1.0.downsample.1.running_mean
layer1.0.downsample.1.running_var
layer1.0.downsample.1.num_batches_tracked
layer1.1.bn1.running_mean
layer1.1.bn1.running_var
layer1.1.bn1.num_batches_tracked
layer1.1.bn2.running_mean
layer1.1.bn2.running_var
layer1.1.bn2.num_batches_tracked
layer1.1.bn3.running_mean
layer1.1.bn3.running_var
layer1.1.bn3.num_batches_tracked
layer1.2.bn1.running_mean
layer1.2.bn1.running_var
layer1.2.bn1.num_batches_tracked
layer1.2.bn2.running_mean
layer1.2.bn2.running_var
layer1.2.bn2.num_batches_tracked
layer1.2.bn3.running_mean
layer1.2.bn3.running_var
layer1.2.bn3.num_batches_tracked
layer2.0.bn1.running_mean
layer2.0.bn1.running_var
layer2.0.bn1.num_batches_

<br>
<br>


# `#03: Named Modules`

<br>
<br>

In [11]:

for name,mod in model.named_modules():
    print(name)


conv1
bn1
relu
maxpool
layer1
layer1.0
layer1.0.conv1
layer1.0.bn1
layer1.0.conv2
layer1.0.bn2
layer1.0.conv3
layer1.0.bn3
layer1.0.relu
layer1.0.downsample
layer1.0.downsample.0
layer1.0.downsample.1
layer1.1
layer1.1.conv1
layer1.1.bn1
layer1.1.conv2
layer1.1.bn2
layer1.1.conv3
layer1.1.bn3
layer1.1.relu
layer1.2
layer1.2.conv1
layer1.2.bn1
layer1.2.conv2
layer1.2.bn2
layer1.2.conv3
layer1.2.bn3
layer1.2.relu
layer2
layer2.0
layer2.0.conv1
layer2.0.bn1
layer2.0.conv2
layer2.0.bn2
layer2.0.conv3
layer2.0.bn3
layer2.0.relu
layer2.0.downsample
layer2.0.downsample.0
layer2.0.downsample.1
layer2.1
layer2.1.conv1
layer2.1.bn1
layer2.1.conv2
layer2.1.bn2
layer2.1.conv3
layer2.1.bn3
layer2.1.relu
layer2.2
layer2.2.conv1
layer2.2.bn1
layer2.2.conv2
layer2.2.bn2
layer2.2.conv3
layer2.2.bn3
layer2.2.relu
layer2.3
layer2.3.conv1
layer2.3.bn1
layer2.3.conv2
layer2.3.bn2
layer2.3.conv3
layer2.3.bn3
layer2.3.relu
layer3
layer3.0
layer3.0.conv1
layer3.0.bn1
layer3.0.conv2
layer3.0.bn2
layer3.0.conv

<br>
<br>

# `#04: Freeze and Unfreeze Things:`

<br>
<br>


In [12]:



# list weight and bias:(all the things are trainbale)
# it's usefull when we need finetuning:
for name,mod in model.named_parameters():
    print(name)
    

# we wanna feeze: conv1 and layer1: 
for name,param in model.named_parameters():
    if "conv1" in name or "layer1" in name:
        param.requires_grad = False 
        

# how to check which is trainable or which is not?
print("=================Show the trainable parameters list===================")
for name,param in model.named_parameters():
    print(f"{name:50} | Trainable: {param.requires_grad}")


# now i want layer: i layer 3 -> bn will be false:
for name,param in model.named_parameters():
    if "layer3" in name:
        if "bn1" in name or "bn2" in name or "bn3" in name:
            param.requires_grad = False 

# how to check which is trainable or which is not?
print("=================Show the trainable parameters list again ===================")
for name,param in model.named_parameters():
    print(f"{name:50} | Trainable: {param.requires_grad}")





conv1.weight
bn1.weight
bn1.bias
layer1.0.conv1.weight
layer1.0.bn1.weight
layer1.0.bn1.bias
layer1.0.conv2.weight
layer1.0.bn2.weight
layer1.0.bn2.bias
layer1.0.conv3.weight
layer1.0.bn3.weight
layer1.0.bn3.bias
layer1.0.downsample.0.weight
layer1.0.downsample.1.weight
layer1.0.downsample.1.bias
layer1.1.conv1.weight
layer1.1.bn1.weight
layer1.1.bn1.bias
layer1.1.conv2.weight
layer1.1.bn2.weight
layer1.1.bn2.bias
layer1.1.conv3.weight
layer1.1.bn3.weight
layer1.1.bn3.bias
layer1.2.conv1.weight
layer1.2.bn1.weight
layer1.2.bn1.bias
layer1.2.conv2.weight
layer1.2.bn2.weight
layer1.2.bn2.bias
layer1.2.conv3.weight
layer1.2.bn3.weight
layer1.2.bn3.bias
layer2.0.conv1.weight
layer2.0.bn1.weight
layer2.0.bn1.bias
layer2.0.conv2.weight
layer2.0.bn2.weight
layer2.0.bn2.bias
layer2.0.conv3.weight
layer2.0.bn3.weight
layer2.0.bn3.bias
layer2.0.downsample.0.weight
layer2.0.downsample.1.weight
layer2.0.downsample.1.bias
layer2.1.conv1.weight
layer2.1.bn1.weight
layer2.1.bn1.bias
layer2.1.conv2.we

In [19]:



def freeze_bn(m):
    if isinstance(m, torch.nn.BatchNorm2d):
        m.eval()
        for p in m.parameters():
            p.requires_grad = False 

model.apply(freeze_bn)

for name,param in model.named_parameters():
    print(f"Name: {name:50} | Trainable? {param.requires_grad}")



Name: conv1.weight                                       | Trainable? True
Name: bn1.weight                                         | Trainable? False
Name: bn1.bias                                           | Trainable? False
Name: layer1.0.conv1.weight                              | Trainable? True
Name: layer1.0.bn1.weight                                | Trainable? False
Name: layer1.0.bn1.bias                                  | Trainable? False
Name: layer1.0.conv2.weight                              | Trainable? True
Name: layer1.0.bn2.weight                                | Trainable? False
Name: layer1.0.bn2.bias                                  | Trainable? False
Name: layer1.0.conv3.weight                              | Trainable? True
Name: layer1.0.bn3.weight                                | Trainable? False
Name: layer1.0.bn3.bias                                  | Trainable? False
Name: layer1.0.downsample.0.weight                       | Trainable? True
Name: layer1.0.do

In [20]:

import torch.optim as optim

# No need memory for freeze parameters:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters())
)


<br>
<br>
<br>


# `#05: Interpret another model name: RegNet`

<br>
<br>
<br>

In [35]:

import warnings
from torchinfo import summary
from torchvision.models import get_model, get_model_weights

warnings.filterwarnings('ignore')

model_name = "regnet_y_1_6gf"
weights = get_model_weights(model_name)
model = get_model(model_name,weights=weights)

print("==================================With Depth:1==================================")
print(summary(model,depth=1))


print()
print("==================================With Depth:2==================================")
print(summary(model,depth=10))



==================================With Depth:1==================================
Layer (type:depth-idx)                                  Param #
RegNet                                                  --
├─SimpleStemIN: 1-1                                     928
├─Sequential: 1-2                                       10,312,502
├─AdaptiveAvgPool2d: 1-3                                --
├─Linear: 1-4                                           889,000
Total params: 11,202,430
Trainable params: 11,202,430
Non-trainable params: 0

==================================With Depth:2==================================
Layer (type:depth-idx)                                  Param #
RegNet                                                  --
├─SimpleStemIN: 1-1                                     --
│    └─Conv2d: 2-1                                      864
│    └─BatchNorm2d: 2-2                                 64
│    └─ReLU: 2-3                                        --
├─Sequential: 1-2         

In [37]:


for name, module in model.named_modules():
    print(name)
    
    


stem
stem.0
stem.1
stem.2
trunk_output
trunk_output.block1
trunk_output.block1.block1-0
trunk_output.block1.block1-0.proj
trunk_output.block1.block1-0.proj.0
trunk_output.block1.block1-0.proj.1
trunk_output.block1.block1-0.f
trunk_output.block1.block1-0.f.a
trunk_output.block1.block1-0.f.a.0
trunk_output.block1.block1-0.f.a.1
trunk_output.block1.block1-0.f.a.2
trunk_output.block1.block1-0.f.b
trunk_output.block1.block1-0.f.b.0
trunk_output.block1.block1-0.f.b.1
trunk_output.block1.block1-0.f.b.2
trunk_output.block1.block1-0.f.se
trunk_output.block1.block1-0.f.se.avgpool
trunk_output.block1.block1-0.f.se.fc1
trunk_output.block1.block1-0.f.se.fc2
trunk_output.block1.block1-0.f.se.activation
trunk_output.block1.block1-0.f.se.scale_activation
trunk_output.block1.block1-0.f.c
trunk_output.block1.block1-0.f.c.0
trunk_output.block1.block1-0.f.c.1
trunk_output.block1.block1-0.activation
trunk_output.block1.block1-1
trunk_output.block1.block1-1.f
trunk_output.block1.block1-1.f.a
trunk_output.b

In [ ]:



for name, module in model.named_buffers():
    print(name)
    
    

stem.1.running_mean
stem.1.running_var
stem.1.num_batches_tracked
trunk_output.block1.block1-0.proj.1.running_mean
trunk_output.block1.block1-0.proj.1.running_var
trunk_output.block1.block1-0.proj.1.num_batches_tracked
trunk_output.block1.block1-0.f.a.1.running_mean
trunk_output.block1.block1-0.f.a.1.running_var
trunk_output.block1.block1-0.f.a.1.num_batches_tracked
trunk_output.block1.block1-0.f.b.1.running_mean
trunk_output.block1.block1-0.f.b.1.running_var
trunk_output.block1.block1-0.f.b.1.num_batches_tracked
trunk_output.block1.block1-0.f.c.1.running_mean
trunk_output.block1.block1-0.f.c.1.running_var
trunk_output.block1.block1-0.f.c.1.num_batches_tracked
trunk_output.block1.block1-1.f.a.1.running_mean
trunk_output.block1.block1-1.f.a.1.running_var
trunk_output.block1.block1-1.f.a.1.num_batches_tracked
trunk_output.block1.block1-1.f.b.1.running_mean
trunk_output.block1.block1-1.f.b.1.running_var
trunk_output.block1.block1-1.f.b.1.num_batches_tracked
trunk_output.block1.block1-1.f.

In [44]:


for name, param in model.named_parameters():
    print(f"{name:50} | Trainable?: {param.requires_grad}")
    
    

stem.0.weight                                      | Trainable?: True
stem.1.weight                                      | Trainable?: True
stem.1.bias                                        | Trainable?: True
trunk_output.block1.block1-0.proj.0.weight         | Trainable?: True
trunk_output.block1.block1-0.proj.1.weight         | Trainable?: True
trunk_output.block1.block1-0.proj.1.bias           | Trainable?: True
trunk_output.block1.block1-0.f.a.0.weight          | Trainable?: True
trunk_output.block1.block1-0.f.a.1.weight          | Trainable?: True
trunk_output.block1.block1-0.f.a.1.bias            | Trainable?: True
trunk_output.block1.block1-0.f.b.0.weight          | Trainable?: True
trunk_output.block1.block1-0.f.b.1.weight          | Trainable?: True
trunk_output.block1.block1-0.f.b.1.bias            | Trainable?: True
trunk_output.block1.block1-0.f.se.fc1.weight       | Trainable?: True
trunk_output.block1.block1-0.f.se.fc1.bias         | Trainable?: True
trunk_output.block1.